<a href="https://colab.research.google.com/github/tipismaca/Trabajo-Bebidas-del-Maipo-SA/blob/main/Codigo/Entrega2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Caso de Estudio: Bebidas del Maipo S.A. — Entrega 2
**Autoras:** Maytte Almuna, Macarena Campos, Darlyn Jerez y Valentina Roa


## 1. Configuración inicial
Clonación del repositorio y librerías

In [5]:
# Clona el repositorio para tener acceso a los datos
!rm -rf Trabajo-Bebidas-del-Maipo-SA
!git clone https://github.com/tipismaca/Trabajo-Bebidas-del-Maipo-SA.git

Cloning into 'Trabajo-Bebidas-del-Maipo-SA'...
remote: Enumerating objects: 575, done.
remote: Counting objects: 100% (105/105), done.
remote: Compressing objects: 100% (70/70), done.
remote: Total 575 (delta 85), reused 35 (delta 35), pack-reused 470 (from 1)
Receiving objects: 100% (575/575), 35.65 MiB | 27.87 MiB/s, done.
Resolving deltas: 100% (308/308), done.


In [6]:
%pip install -q "pulp>=2.7,<3"   # PuLP 2.x trae CBC incluido
!pip install -q highspy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pulp
from IPython.display import display


## 2. Carga de datos

In [7]:
ruta_base = "Trabajo-Bebidas-del-Maipo-SA/Datos_crudos/"
ruta_resultados = "Trabajo-Bebidas-del-Maipo-SA/Resultados/"

# ================================================================
# CARGA DE ARCHIVOS Y RESULTADOS DE LA ENTREGA 1
# Ejecutar antes de A1
# ================================================================

# Datos de la Entrega 2
maestro = pd.read_csv(ruta_base + "maestro_sku.csv")
inv_inicial = pd.read_csv(ruta_base + "inventario_inicial.csv")
capacidad = pd.read_csv(ruta_base + "capacidad_horizonte.csv")
setups = pd.read_csv(ruta_base + "setups_cip.csv")
compromisos = pd.read_csv(
    ruta_base + "compromisos_semana_focal.csv"
)

# Resultados heredados de la Entrega 1
tabla_c1 = pd.read_csv(
    ruta_resultados + "merma_por_linea.csv"
)

tabla_c2 = pd.read_csv(
    ruta_resultados + "sobrellenado_por_formato.csv"
)

tabla_c3 = pd.read_csv(
    ruta_resultados + "pronostico_sku_horizonte.csv"
)

tabla_c4 = pd.read_csv(
    ruta_resultados + "sigma_error_sku_horizonte.csv"
)

# Merma por línea
rho = dict(zip(
    tabla_c1["Línea"],
    tabla_c1["ρ (proporción)"].astype(float)
))

# Sobrellenado: litros adicionales por caja y formato
jarabe_extra_L_caja = dict(zip(
    tabla_c2["Formato (L)"].astype(float),
    tabla_c2["Jarabe adicional (L/caja)"].astype(float)
))

# Convertir las tablas de pronóstico y error a formato largo
def matriz_a_largo(tabla, nombre_valor):

    columnas_h = [
        f"h={t}" for t in range(1, 14)
    ]

    largo = tabla.melt(
        id_vars=["Familia", "SKU"],
        value_vars=columnas_h,
        var_name="h",
        value_name=nombre_valor
    )

    largo["h"] = (
        largo["h"]
        .str.replace("h=", "", regex=False)
        .astype(int)
    )

    largo[nombre_valor] = pd.to_numeric(
        largo[nombre_valor],
        errors="raise"
    )

    return largo.rename(columns={
        "SKU": "sku",
        "Familia": "familia"
    })


pronostico = matriz_a_largo(tabla_c3, "d_hat")
sigma_err = matriz_a_largo(tabla_c4, "sigma")

if pronostico.duplicated(["sku", "h"]).any():
    raise ValueError("Hay pronósticos duplicados por SKU y semana.")

if sigma_err.duplicated(["sku", "h"]).any():
    raise ValueError("Hay errores de pronóstico duplicados por SKU y semana.")

# Demanda utilizada en el modelo
d_hat = {
    (fila.sku, int(fila.h)): float(fila.d_hat)
    for fila in pronostico.itertuples(index=False)
}

print("Archivos cargados correctamente.")
print(f"SKU del maestro: {maestro['sku'].nunique()}")
print(f"Semanas del horizonte: {capacidad['t'].nunique()}")
print(f"Pronósticos SKU-semana: {len(d_hat)}")


Archivos cargados correctamente.
SKU del maestro: 9
Semanas del horizonte: 13
Pronósticos SKU-semana: 117


---
## Parte A1. Balance de inventario por sku y semana.


In [8]:
# ================================================================
# A1. BALANCE DE INVENTARIO POR SKU Y SEMANA
# Producción, rendimiento, inventario y atrasos penalizados
# ================================================================
#
# Ejecutar después de la carga de datos.
#
# Variables:
# x[i][t] = producción bruta del SKU i en la semana t, en cajas.
# s[i][t] = inventario vendible al finalizar la semana t, en cajas.
# b[i][t] = demanda pendiente al finalizar la semana t, en cajas.
#
# Rendimiento:
# rend[i] = 1 - rho de la línea correspondiente.
#
# Balance:
# s_anterior - b_anterior + rend[i] * x[i][t]
#     = demanda[i,t] + s[i][t] - b[i][t]
#
# Este bloque construye el modelo.
# La resolución final se realizará después de incorporar A2–A8.
# ================================================================


# ----------------------------------------------------------------
# 1. COMPROBAR QUE LA CARGA DE DATOS ESTÁ DISPONIBLE
# ----------------------------------------------------------------

necesarios = [
    "maestro",
    "inv_inicial",
    "capacidad",
    "rho",
    "d_hat"
]

faltantes = [nombre for nombre in necesarios if nombre not in globals()]

if faltantes:
    raise ValueError(
        "Primero ejecuta la carga de datos. Faltan: "
        + ", ".join(faltantes)
    )


# ----------------------------------------------------------------
# 2. CONJUNTOS DEL MODELO
# ----------------------------------------------------------------

if maestro["sku"].duplicated().any():
    raise ValueError("Hay SKU repetidos en maestro_sku.csv.")

if inv_inicial["sku"].duplicated().any():
    raise ValueError("Hay SKU repetidos en inventario_inicial.csv.")

if capacidad["t"].duplicated().any():
    raise ValueError("Hay semanas repetidas en capacidad_horizonte.csv.")

SKUS = maestro["sku"].tolist()

LINEAS = sorted(
    maestro["linea_envasado"].unique().tolist()
)

T = sorted(
    capacidad["t"].astype(int).tolist()
)

if T != list(range(1, 14)):
    raise ValueError("El horizonte debe contener las semanas 1 a 13.")

if set(inv_inicial["sku"]) != set(SKUS):
    raise ValueError(
        "Los SKU del inventario inicial no coinciden con el maestro."
    )

# Unir los datos por SKU, no por la posición de las filas.
datos = maestro.set_index("sku").join(
    inv_inicial.set_index("sku"),
    how="left",
    validate="one_to_one"
)

cap = capacidad.set_index("t").sort_index()

linea_de = datos["linea_envasado"].to_dict()
familia_de = datos["familia"].to_dict()

FAMILIAS = sorted(
    datos["familia"].unique().tolist()
)

# Estos conjuntos también se utilizarán en los siguientes bloques.
skus_linea = {
    linea: [
        i for i in SKUS
        if linea_de[i] == linea
    ]
    for linea in LINEAS
}

skus_fam = {
    familia: [
        i for i in SKUS
        if familia_de[i] == familia
    ]
    for familia in FAMILIAS
}

skus_linea_familia = {
    (linea, familia): [
        i for i in SKUS
        if linea_de[i] == linea
        and familia_de[i] == familia
    ]
    for linea in LINEAS
    for familia in FAMILIAS
}


# ----------------------------------------------------------------
# 3. PARÁMETROS: DATOS Y SUPUESTOS
# ----------------------------------------------------------------

# Fuente: maestro_sku.csv.
# Costo por caja bruta producida.
p_var = (
    datos["costo_variable_caja_clp"]
    .astype(float)
    .to_dict()
)

# Fuente: maestro_sku.csv y enunciado, sección 2.4.
# CLP por caja de inventario al cierre de cada semana.
h = (
    datos["costo_inventario_caja_semana_clp"]
    .astype(float)
    .to_dict()
)

# Fuente: maestro_sku.csv y enunciado, sección 2.4.
# CLP por caja pendiente al cierre de cada semana.
pi_atr = (
    datos["penalidad_atraso_caja_semana_clp"]
    .astype(float)
    .to_dict()
)

# Fuente: inventario_inicial.csv.
# Son cajas de producto terminado vendible.
s0 = (
    datos["inventario_inicial_cajas"]
    .astype(float)
    .to_dict()
)

# Supuesto: no hay pedidos atrasados antes de la semana 1.
b0 = {i: 0.0 for i in SKUS}

# Fuente: pronóstico de la Entrega 1.
# No se altera ni se recorta el pronóstico en este bloque.
combinaciones = {
    (i, t)
    for i in SKUS
    for t in T
}

faltan_pronosticos = combinaciones - set(d_hat.keys())

if faltan_pronosticos:
    raise ValueError(
        "Faltan pronósticos para algunas combinaciones SKU-semana: "
        + str(sorted(faltan_pronosticos))
    )

d = {
    (i, t): float(d_hat[i, t])
    for i in SKUS
    for t in T
}

# Fuente: merma por línea de la Entrega 1.
# rho representa la fracción defectuosa/no vendible.
# El rendimiento vendible es 1 - rho.
if not set(LINEAS).issubset(rho.keys()):
    raise ValueError("Falta la merma de alguna línea en rho.")

rend = {
    i: 1.0 - float(rho[linea_de[i]])
    for i in SKUS
}


# ----------------------------------------------------------------
# 4. VALIDACIONES DE LOS PARÁMETROS
# ----------------------------------------------------------------

for nombre, parametro in [
    ("Costo variable", p_var),
    ("Costo de inventario", h),
    ("Penalidad por atraso", pi_atr),
    ("Inventario inicial", s0),
    ("Demanda", d)
]:
    valores = np.array(list(parametro.values()), dtype=float)

    if not np.isfinite(valores).all():
        raise ValueError(
            f"{nombre}: existen valores vacíos o no finitos."
        )

    if (valores < 0).any():
        raise ValueError(
            f"{nombre}: existen valores negativos."
        )

for i in SKUS:
    if not np.isfinite(rend[i]) or not 0 < rend[i] <= 1:
        raise ValueError(
            f"Rendimiento inválido para {i}: {rend[i]}"
        )

# Comprobación contra los costos indicados en el enunciado.
# No se reemplazan silenciosamente los datos si existe una diferencia.
if not np.allclose(list(h.values()), 95.0):
    raise ValueError(
        "El costo de inventario del archivo no coincide con "
        "los $95 por caja-semana indicados en el enunciado."
    )

if not np.allclose(list(pi_atr.values()), 1400.0):
    raise ValueError(
        "La penalidad del archivo no coincide con los "
        "$1.400 por caja-semana indicados en el enunciado."
    )


# ----------------------------------------------------------------
# 5. CREAR UN ÚNICO MODELO PARA A1–A8
# ----------------------------------------------------------------

# Volver a ejecutar A1 reinicia el modelo.
# Después deben ejecutarse nuevamente A2–A8 en orden.
modelo = pulp.LpProblem(
    "BdM_LotSizing_13_Semanas",
    pulp.LpMinimize
)

# Supuesto de planificación táctica:
# las cantidades se representan como cajas equivalentes continuas.
# No se redondean después de resolver, porque eso podría violar
# balances, capacidades o compromisos.

x = pulp.LpVariable.dicts(
    "produccion_bruta",
    (SKUS, T),
    lowBound=0,
    cat="Continuous"
)

s = pulp.LpVariable.dicts(
    "inventario_final",
    (SKUS, T),
    lowBound=0,
    cat="Continuous"
)

b = pulp.LpVariable.dicts(
    "atraso_final",
    (SKUS, T),
    lowBound=0,
    cat="Continuous"
)


# ----------------------------------------------------------------
# 6. BALANCE POR SKU Y SEMANA
# ----------------------------------------------------------------

for i in SKUS:
    for t in T:

        inventario_anterior = (
            s0[i] if t == 1 else s[i][t - 1]
        )

        atraso_anterior = (
            b0[i] if t == 1 else b[i][t - 1]
        )

        modelo += (
            inventario_anterior
            - atraso_anterior
            + rend[i] * x[i][t]
            ==
            d[i, t]
            + s[i][t]
            - b[i][t],
            f"balance_{i}_{t}"
        )

# La demanda que no se atiende permanece como atraso.
# No desaparece ni se considera automáticamente una venta perdida.
#
# La merma se aplica únicamente a la producción nueva:
# el inventario inicial ya corresponde a producto vendible.


# ----------------------------------------------------------------
# 7. COMPONENTES DE COSTO DE A1
# ----------------------------------------------------------------

# Los siguientes bloques agregarán los costos de preparación,
# limpieza y horas extra a este mismo diccionario.

costos = {
    "Producción": pulp.lpSum(
        p_var[i] * x[i][t]
        for i in SKUS
        for t in T
    ),

    "Inventario": pulp.lpSum(
        h[i] * s[i][t]
        for i in SKUS
        for t in T
    ),

    "Atrasos": pulp.lpSum(
        pi_atr[i] * b[i][t]
        for i in SKUS
        for t in T
    )
}

# Objetivo parcial: se actualizará con TODOS los componentes
# antes de resolver el modelo completo al finalizar A8.
modelo.setObjective(
    pulp.lpSum(costos.values())
)


# ----------------------------------------------------------------
# 8. TABLAS COMPACTAS DE LOS DATOS UTILIZADOS
# ----------------------------------------------------------------

def mostrar_tabla(titulo, tabla, formatos=None):
    print(titulo)

    display(
        tabla.style
        .hide(axis="index")
        .format(
            formatos if formatos is not None else {},
            thousands=".",
            decimal=",",
            na_rep="—"
        )
        .set_properties(**{
            "text-align": "center",
            "padding": "4px 6px",
            "border": "1px solid #888"
        })
        .set_table_styles([
            {
                "selector": "th",
                "props": [
                    ("text-align", "center"),
                    ("padding", "4px 6px"),
                    ("border", "1px solid #888")
                ]
            },
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("width", "auto")
                ]
            }
        ])
    )


tabla_parametros_a1 = pd.DataFrame({
    "SKU": SKUS,

    "Línea": [
        linea_de[i] for i in SKUS
    ],

    "Inventario inicial (cajas)": [
        s0[i] for i in SKUS
    ],

    "Demanda 13 semanas (cajas)": [
        sum(d[i, t] for t in T)
        for i in SKUS
    ],

    "Costo variable ($/caja)": [
        p_var[i] for i in SKUS
    ]
})

mostrar_tabla(
    "A1 — Datos de entrada por SKU",
    tabla_parametros_a1,
    {
        "Inventario inicial (cajas)": "{:,.0f}",
        "Demanda 13 semanas (cajas)": "{:,.0f}",
        "Costo variable ($/caja)": "{:,.0f}"
    }
)


tabla_rendimientos_a1 = pd.DataFrame({
    "Línea": LINEAS,

    "Merma (%)": [
        100 * float(rho[linea])
        for linea in LINEAS
    ],

    "Rendimiento vendible (%)": [
        100 * (1 - float(rho[linea]))
        for linea in LINEAS
    ]
})

mostrar_tabla(
    "A1 — Merma y rendimiento heredados de la Entrega 1",
    tabla_rendimientos_a1,
    {
        "Merma (%)": "{:.2f}",
        "Rendimiento vendible (%)": "{:.2f}"
    }
)


tabla_costos_a1 = pd.DataFrame({
    "Concepto": [
        "Inventario",
        "Demanda atrasada"
    ],
    "Costo ($/caja/semana)": [
        95,
        1400
    ],
    "Aplicación": [
        "Inventario al cierre de cada semana",
        "Cajas pendientes al cierre de cada semana"
    ]
})

mostrar_tabla(
    "A1 — Costos de inventario y atraso",
    tabla_costos_a1,
    {
        "Costo ($/caja/semana)": "{:,.0f}"
    }
)


# ----------------------------------------------------------------
# 9. CONFIRMACIÓN DE CONSTRUCCIÓN
# ----------------------------------------------------------------

numero_balances = sum(
    nombre.startswith("balance_")
    for nombre in modelo.constraints
)

assert numero_balances == len(SKUS) * len(T)

print(
    f"A1 construido: {len(SKUS)} SKU, {len(T)} semanas "
    f"y {numero_balances} balances."
)

print(
    "Supuestos: atraso inicial cero y cantidades expresadas "
    "en cajas equivalentes continuas."
)

print(
    "Todavía no se ha resuelto el modelo. "
    "Los resultados finales se calcularán al completar A2–A8."
)

A1 — Datos de entrada por SKU


SKU,Línea,Inventario inicial (cajas),Demanda 13 semanas (cajas),Costo variable ($/caja)
NEC-D05,L1,1.024,13.875,3.840
NEC-D15,L2,3.162,36.281,5.760
NEC-D30,L2,339,6.851,7.680
CIT-L05,L1,1.217,30.808,2.820
CIT-L15,L2,1.282,23.257,4.230
CIT-L30,L2,255,5.625,5.640
FUN-B05,L1,2.135,29.659,4.860
FUN-B15,L2,1.261,18.486,7.290
FUN-B30,L2,155,2.509,9.720


A1 — Merma y rendimiento heredados de la Entrega 1


Línea,Merma (%),Rendimiento vendible (%)
L1,"1,92","98,08"
L2,"3,11","96,89"


A1 — Costos de inventario y atraso


Concepto,Costo ($/caja/semana),Aplicación
Inventario,95,Inventario al cierre de cada semana
Demanda atrasada,1.400,Cajas pendientes al cierre de cada semana


A1 construido: 9 SKU, 13 semanas y 117 balances.
Supuestos: atraso inicial cero y cantidades expresadas en cajas equivalentes continuas.
Todavía no se ha resuelto el modelo. Los resultados finales se calcularán al completar A2–A8.


---
##Parte A2. Variables binarias de preparación y su enlace xit ≤ Mit yi

In [9]:
# ================================================================
# A2. VARIABLES BINARIAS DE PREPARACIÓN Y COTA M JUSTIFICADA
# ================================================================
#
# y[i][t] = 1 si se habilita la producción del SKU i en la semana t.
# y[i][t] = 0 si no se habilita.
#
# Enlace:
# x[i][t] <= M[i,t] * y[i][t]
#
# Cota física:
# M[i,t] = tasa de envasado del SKU
#          * (horas regulares + máximo de horas extra de su línea).
#
# Esta cota representa el máximo individual si el SKU dispusiera
# de todas esas horas, sin descontar preparaciones.
#
# La capacidad compartida y los tiempos de preparación se
# incorporarán en A3. Las horas extra se decidirán en A5.
# ================================================================


# ----------------------------------------------------------------
# 1. VERIFICAR QUE A1 ESTÁ CONSTRUIDO
# ----------------------------------------------------------------

necesarios_a2 = [
    "modelo", "datos", "cap", "SKUS", "LINEAS", "T",
    "linea_de", "x", "mostrar_tabla"
]

faltantes_a2 = [
    nombre for nombre in necesarios_a2
    if nombre not in globals()
]

if faltantes_a2:
    raise ValueError(
        "Primero ejecuta A1. Faltan: "
        + ", ".join(faltantes_a2)
    )

# Evitar incorporar dos veces las mismas variables y restricciones.
if any(
    nombre.startswith("enlace_")
    for nombre in modelo.constraints
):
    raise ValueError(
        "A2 ya está incorporado. Para reiniciar, ejecuta A1 "
        "y luego los bloques siguientes en orden."
    )


# ----------------------------------------------------------------
# 2. TASAS DE ENVASADO
# ----------------------------------------------------------------

# Fuente: maestro_sku.csv.
# El enunciado indica:
# L1, formato 0,5 L: 180 cajas/hora.
# L2, formato 1,5 L: 210 cajas/hora.
# L2, formato 3,0 L: 130 cajas/hora.

tasa = (
    datos["tasa_envasado_cajas_h"]
    .astype(float)
    .to_dict()
)

for i in SKUS:
    if not np.isfinite(tasa[i]) or tasa[i] <= 0:
        raise ValueError(
            f"La tasa de envasado de {i} debe ser positiva."
        )


# ----------------------------------------------------------------
# 3. HORAS DISPONIBLES POR LÍNEA Y SEMANA
# ----------------------------------------------------------------

# Fuente: capacidad_horizonte.csv.
#
# Se utilizan las horas específicas de cada semana.
# Las horas regulares del archivo ya incorporan los ajustes
# de disponibilidad: no se resta nuevamente la mantención.
#
# El máximo de horas extra se usa aquí solamente para calcular M.
# Esto NO obliga a utilizar esas horas ni las deja gratuitas.

H_reg_linea = {
    (linea, t): float(
        cap.loc[t, f"horas_{linea}_regular"]
    )
    for linea in LINEAS
    for t in T
}

H_extra_max = {
    (linea, t): float(
        cap.loc[t, f"horas_{linea}_extra_max"]
    )
    for linea in LINEAS
    for t in T
}

for nombre, parametro in [
    ("Horas regulares", H_reg_linea),
    ("Máximo de horas extra", H_extra_max)
]:
    valores = np.array(
        list(parametro.values()),
        dtype=float
    )

    if not np.isfinite(valores).all():
        raise ValueError(
            f"{nombre}: hay datos vacíos o no finitos."
        )

    if (valores < 0).any():
        raise ValueError(
            f"{nombre}: hay valores negativos."
        )


# ----------------------------------------------------------------
# 4. CALCULAR M POR SKU Y SEMANA
# ----------------------------------------------------------------

# Unidades:
# (cajas/hora) * (horas/semana) = cajas/semana.
#
# M se expresa en producción BRUTA, igual que x.
# No se multiplica por el rendimiento vendible.
#
# No se limita M a la demanda de esa semana, porque el modelo
# puede producir anticipadamente para mantener inventario,
# cubrir reservas o atender demanda atrasada.

M = {
    (i, t): tasa[i] * (
        H_reg_linea[linea_de[i], t]
        + H_extra_max[linea_de[i], t]
    )
    for i in SKUS
    for t in T
}


# ----------------------------------------------------------------
# 5. VARIABLE BINARIA DE PREPARACIÓN
# ----------------------------------------------------------------

y = pulp.LpVariable.dicts(
    "preparacion",
    (SKUS, T),
    cat="Binary"
)


# ----------------------------------------------------------------
# 6. ENLACE ENTRE PRODUCCIÓN Y PREPARACIÓN
# ----------------------------------------------------------------

for i in SKUS:
    for t in T:

        modelo += (
            x[i][t] <= M[i, t] * y[i][t],
            f"enlace_{i}_{t}"
        )

# Si y = 0, necesariamente x = 0.
# Si x > 0, necesariamente y = 1.
#
# Por ahora, y = 1 permite producir, pero no obliga a hacerlo.
# En A6, el lote mínimo completará el enlace:
#
# x[i][t] >= lote_minimo[i] * y[i][t]


# ----------------------------------------------------------------
# 7. COSTOS DE PREPARACIÓN
# ----------------------------------------------------------------

# No se agrega un costo fijo arbitrario por activar cada SKU.
#
# Los $38.000 del enunciado corresponden a un CAMBIO DE FORMATO,
# no automáticamente a cada y[i][t] = 1.
#
# Los costos y tiempos de cambios de formato y limpiezas
# deben incorporarse junto con su representación en A3.


# ----------------------------------------------------------------
# 8. TABLA RESUMEN DE LAS COTAS
# ----------------------------------------------------------------

# M puede variar entre semanas si cambia la disponibilidad.
# Se muestra su mínimo y máximo para mantener la tabla compacta.

tabla_a2 = pd.DataFrame({
    "SKU": SKUS,

    "Línea": [
        linea_de[i] for i in SKUS
    ],

    "Tasa (cajas/h)": [
        tasa[i] for i in SKUS
    ],

    "M mínimo (cajas/semana)": [
        min(M[i, t] for t in T)
        for i in SKUS
    ],

    "M máximo (cajas/semana)": [
        max(M[i, t] for t in T)
        for i in SKUS
    ]
})

mostrar_tabla(
    "A2 — Cotas físicas para habilitar la producción",
    tabla_a2,
    {
        "Tasa (cajas/h)": "{:,.0f}",
        "M mínimo (cajas/semana)": "{:,.0f}",
        "M máximo (cajas/semana)": "{:,.0f}"
    }
)


# ----------------------------------------------------------------
# 9. CONFIRMACIÓN DE CONSTRUCCIÓN
# ----------------------------------------------------------------

numero_enlaces = sum(
    nombre.startswith("enlace_")
    for nombre in modelo.constraints
)

assert numero_enlaces == len(SKUS) * len(T)

print(
    f"A2 incorporado: {len(SKUS) * len(T)} variables binarias "
    f"y {numero_enlaces} restricciones de enlace."
)

print(
    "M se calculó con tasas y horas disponibles de los archivos; "
    "no se utilizó un número arbitrario."
)

print(
    "El modelo todavía no se resuelve. "
    "La capacidad compartida se incorpora en A3."
)

A2 — Cotas físicas para habilitar la producción


SKU,Línea,Tasa (cajas/h),M mínimo (cajas/semana),M máximo (cajas/semana)
NEC-D05,L1,180,7.920,8.640
NEC-D15,L2,210,11.760,13.440
NEC-D30,L2,130,7.280,8.320
CIT-L05,L1,180,7.920,8.640
CIT-L15,L2,210,11.760,13.440
CIT-L30,L2,130,7.280,8.320
FUN-B05,L1,180,7.920,8.640
FUN-B15,L2,210,11.760,13.440
FUN-B30,L2,130,7.280,8.320


A2 incorporado: 117 variables binarias y 117 restricciones de enlace.
M se calculó con tasas y horas disponibles de los archivos; no se utilizó un número arbitrario.
El modelo todavía no se resuelve. La capacidad compartida se incorpora en A3.




##Parte A3.  Capacidad de envasado

In [11]:
# ================================================================
# A3. CAPACIDAD DE ENVASADO Y MEZCLA
# Preparaciones, CIP y sobrellenado
# ================================================================
#
# Fuentes:
# - maestro_sku.csv: tasas, formatos y litros nominales por caja.
# - capacidad_horizonte.csv: disponibilidad semanal del recurso.
# - Entrega 1: sobrellenado adicional en litros por caja.
# - Enunciado, secciones 2.2–2.4: tiempos y costos de preparación.
#
# Supuestos de planificación:
# - Cada familia se produce en una campaña por línea y semana.
# - Dentro de cada familia se agrupan sus formatos.
# - Se contabilizan las transiciones dentro de cada semana.
# - No se asigna limpieza inicial ni transición entre semanas,
#   al no representar aquí el estado de las líneas entre períodos.
#
# Las horas extra se incorporarán en A5.
# El lead time del jarabe se incorporará en A6.
# Todavía no se resuelve el modelo.
# ================================================================

# Librería estándar de Python: permite enumerar los posibles
# órdenes de las tres familias, sin instalar paquetes adicionales.
from itertools import permutations


# ----------------------------------------------------------------
# 1. VERIFICACIONES
# ----------------------------------------------------------------

necesarios_a3 = [
    "modelo", "datos", "cap", "SKUS", "LINEAS", "FAMILIAS", "T",
    "x", "y", "tasa", "linea_de", "familia_de",
    "skus_linea", "skus_fam", "skus_linea_familia",
    "H_reg_linea", "jarabe_extra_L_caja", "costos",
    "mostrar_tabla"
]

faltantes_a3 = [
    nombre for nombre in necesarios_a3
    if nombre not in globals()
]

if faltantes_a3:
    raise ValueError(
        "Primero ejecuta A1 y A2. Faltan: "
        + ", ".join(faltantes_a3)
    )

if any(
    nombre.startswith("cap_envasado_")
    or nombre.startswith("cap_mezcla_")
    for nombre in modelo.constraints
):
    raise ValueError(
        "A3 ya está incorporado. Para reiniciar, ejecuta A1 "
        "y luego los bloques siguientes en orden."
    )

if set(FAMILIAS) != {"CIT", "FUN", "NEC"}:
    raise ValueError(
        "Revisa los códigos de familia: se esperan CIT, FUN y NEC."
    )

# En este caso existe un SKU por familia y formato.
# Esta condición permite contar los cambios de formato mediante
# el número de SKU activos dentro de cada familia y línea.
if datos.reset_index().duplicated(
    ["linea_envasado", "familia", "formato_L"]
).any():
    raise ValueError(
        "Hay varios SKU para una misma familia, línea y formato. "
        "Se debe adaptar el conteo de cambios de formato."
    )


# ----------------------------------------------------------------
# 2. PARÁMETROS DEL ENVASADO: ENUNCIADO
# ----------------------------------------------------------------

# Horas de CIP: familia saliente -> familia entrante.
cip_envasado = {
    ("CIT", "FUN"): 1.0,
    ("CIT", "NEC"): 1.0,
    ("FUN", "CIT"): 2.4,
    ("FUN", "NEC"): 1.4,
    ("NEC", "CIT"): 2.8,
    ("NEC", "FUN"): 2.8
}

# Cambio de formato dentro de la misma familia.
tiempo_cambio_formato = {
    "L1": 0.4,
    "L2": 0.6
}

costo_cambio_formato = 38000.0       # CLP por cambio
costo_hora_cip = 74000.0            # CLP por hora de CIP

# L1 tiene un solo formato; por ello, el conteo de cambios
# de formato de L1 será cero en este caso.


# ----------------------------------------------------------------
# 3. FAMILIAS ACTIVAS EN CADA LÍNEA Y SEMANA
# ----------------------------------------------------------------

u_env = pulp.LpVariable.dicts(
    "familia_activa_envasado",
    (LINEAS, FAMILIAS, T),
    cat="Binary"
)

linea_activa = pulp.LpVariable.dicts(
    "linea_activa",
    (LINEAS, T),
    cat="Binary"
)

for linea in LINEAS:
    for t in T:

        for familia in FAMILIAS:

            productos = skus_linea_familia[linea, familia]

            # Una familia está activa si se prepara alguno
            # de sus SKU en esa línea y semana.
            for i in productos:
                modelo += (
                    y[i][t] <= u_env[linea][familia][t],
                    f"familia_env_lb_{linea}_{i}_{t}"
                )

            modelo += (
                u_env[linea][familia][t]
                <= pulp.lpSum(y[i][t] for i in productos),
                f"familia_env_ub_{linea}_{familia}_{t}"
            )

            modelo += (
                u_env[linea][familia][t] <= linea_activa[linea][t],
                f"linea_activa_lb_{linea}_{familia}_{t}"
            )

        modelo += (
            linea_activa[linea][t]
            <= pulp.lpSum(
                u_env[linea][familia][t]
                for familia in FAMILIAS
            ),
            f"linea_activa_ub_{linea}_{t}"
        )


# ----------------------------------------------------------------
# 4. POSIBLES ÓRDENES DE LAS FAMILIAS EN ENVASADO
# ----------------------------------------------------------------

# Con tres familias hay pocos órdenes posibles.
# Se incluyen campañas de una, dos o tres familias.
# El modelo elegirá una alternativa compatible con las familias
# activas y contabilizará sus transiciones reales.

secuencias_familias = []

for cantidad in range(1, len(FAMILIAS) + 1):
    secuencias_familias.extend(
        permutations(FAMILIAS, cantidad)
    )

PATRONES = list(range(len(secuencias_familias)))

horas_cip_patron = {}

for k in PATRONES:

    secuencia = secuencias_familias[k]

    horas_cip_patron[k] = sum(
        cip_envasado[secuencia[j], secuencia[j + 1]]
        for j in range(len(secuencia) - 1)
    )

patron_env = pulp.LpVariable.dicts(
    "patron_familias_envasado",
    (LINEAS, PATRONES, T),
    cat="Binary"
)

for linea in LINEAS:
    for t in T:

        # Una secuencia si la línea está activa; ninguna si está parada.
        modelo += (
            pulp.lpSum(
                patron_env[linea][k][t]
                for k in PATRONES
            ) == linea_activa[linea][t],
            f"seleccion_patron_{linea}_{t}"
        )

        # La secuencia elegida contiene exactamente las familias activas.
        for familia in FAMILIAS:
            modelo += (
                pulp.lpSum(
                    patron_env[linea][k][t]
                    for k in PATRONES
                    if familia in secuencias_familias[k]
                ) == u_env[linea][familia][t],
                f"contenido_patron_{linea}_{familia}_{t}"
            )


# ----------------------------------------------------------------
# 5. TIEMPOS DE PREPARACIÓN Y CAPACIDAD DE ENVASADO
# ----------------------------------------------------------------

cambios_formato = {}
horas_formato_env = {}
horas_cip_env = {}
horas_preparacion_env = {}
horas_uso_env = {}

for linea in LINEAS:
    for t in T:

        # Dentro de cada familia:
        # un formato activo -> cero cambios;
        # dos formatos activos -> un cambio.
        cambios_formato[linea, t] = (
            pulp.lpSum(
                y[i][t] for i in skus_linea[linea]
            )
            - pulp.lpSum(
                u_env[linea][familia][t]
                for familia in FAMILIAS
            )
        )

        horas_formato_env[linea, t] = (
            tiempo_cambio_formato[linea]
            * cambios_formato[linea, t]
        )

        horas_cip_env[linea, t] = pulp.lpSum(
            horas_cip_patron[k] * patron_env[linea][k][t]
            for k in PATRONES
        )

        horas_preparacion_env[linea, t] = (
            horas_formato_env[linea, t]
            + horas_cip_env[linea, t]
        )

        # Producción bruta / tasa = horas de envasado.
        horas_uso_env[linea, t] = (
            pulp.lpSum(
    (1.0 / tasa[i]) * x[i][t]
    for i in skus_linea[linea]
)
            + horas_preparacion_env[linea, t]
        )

        # A5 reemplazará esta restricción para incorporar
        # las horas extra como variable acotada y con costo.
        modelo += (
            horas_uso_env[linea, t]
            <= H_reg_linea[linea, t],
            f"cap_envasado_{linea}_{t}"
        )


# ----------------------------------------------------------------
# 6. LITROS REQUERIDOS POR CAJA BRUTA
# ----------------------------------------------------------------

# Fuente: maestro_sku.csv + sobrellenado de la Entrega 1.
#
# Cada caja producida consume:
# litros nominales + litros adicionales por sobrellenado.
#
# No se aplica nuevamente 1/(1-rho), porque x ya representa
# producción bruta y la merma ya se incorporó en el balance de A1.

litros_nominales_caja = (
    datos["litros_por_caja"]
    .astype(float)
    .to_dict()
)

formato_de = (
    datos["formato_L"]
    .astype(float)
    .to_dict()
)

litros_caja = {}

for i in SKUS:

    formato = formato_de[i]

    if formato not in jarabe_extra_L_caja:
        raise ValueError(
            f"Falta el sobrellenado del formato {formato} L."
        )

    adicional = float(jarabe_extra_L_caja[formato])

    if not np.isfinite(adicional) or adicional < 0:
        raise ValueError(
            f"Sobrellenado adicional inválido para {i}."
        )

    litros_caja[i] = litros_nominales_caja[i] + adicional

    if not np.isfinite(litros_caja[i]) or litros_caja[i] <= 0:
        raise ValueError(
            f"Litros por caja inválidos para {i}."
        )


# ----------------------------------------------------------------
# 7. DISPONIBILIDAD Y TASA DE MEZCLA
# ----------------------------------------------------------------

# Fuente: capacidad_horizonte.csv.
# Se usan las horas semanales del archivo sin volver a descontar
# las mantenciones programadas.

tasa_mezcla = {
    t: float(cap.loc[t, "tasa_mezcla_L_h"])
    for t in T
}

H_reg_mezcla = {
    t: float(cap.loc[t, "horas_mezcla_regular"])
    for t in T
}

H_extra_max_mezcla = {
    t: float(cap.loc[t, "horas_mezcla_extra_max"])
    for t in T
}

# Enunciado: entre dos familias distintas se requieren 1,2 horas.
horas_cip_mezcla = 1.2

for t in T:

    if not np.isfinite(tasa_mezcla[t]) or tasa_mezcla[t] <= 0:
        raise ValueError(
            f"Tasa de mezcla inválida en la semana {t}."
        )

    for valor in [H_reg_mezcla[t], H_extra_max_mezcla[t]]:
        if not np.isfinite(valor) or valor < 0:
            raise ValueError(
                f"Disponibilidad de mezcla inválida en la semana {t}."
            )


# ----------------------------------------------------------------
# 8. FAMILIAS ACTIVAS Y LIMPIEZAS EN MEZCLA
# ----------------------------------------------------------------

w = pulp.LpVariable.dicts(
    "familia_activa_mezcla",
    (FAMILIAS, T),
    cat="Binary"
)

mez

PulpError: overlapping constraint names: familia_env_lb_L1_CIT_L05_1

 ## Parte A4.  Stock de seguridad

In [ ]:
# ================================================================
# A4. STOCK DE SEGURIDAD POR SKU Y SEMANA
# ================================================================
#
# Fuentes:
# - inventario_inicial.csv:
#       nivel_servicio_objetivo.
# - sigma_error_sku_horizonte.csv, cargado en tabla_c4:
#       desviación estándar del error por SKU y horizonte.
#
# Fórmula:
# SS[i,t] = z_servicio[i] * sigma_ih[i,t]
#
# Restricción:
# s[i][t] >= SS[i,t]
#
# Supuestos:
# 1. Errores de pronóstico aproximadamente normales y sin sesgo.
# 2. El nivel de servicio se interpreta como probabilidad objetivo
#    de que la demanda semanal no exceda el pronóstico más SS.
# 3. Se mantiene una reserva protegida al cierre de cada semana,
#    incluso si existen pedidos pendientes penalizados.
#
# Este bloque NO resuelve todavía el modelo.
# ================================================================


# ----------------------------------------------------------------
# 1. HERRAMIENTA PARA CALCULAR EL CUANTIL NORMAL
# ----------------------------------------------------------------

# statistics pertenece a Python: no requiere instalación.
from statistics import NormalDist


# ----------------------------------------------------------------
# 2. VERIFICAR LOS DATOS Y EL MODELO
# ----------------------------------------------------------------

necesarios_a4 = [
    "modelo", "datos", "SKUS", "T", "s",
    "tabla_c4", "mostrar_tabla"
]

faltantes_a4 = [
    nombre for nombre in necesarios_a4
    if nombre not in globals()
]

if faltantes_a4:
    raise ValueError(
        "Primero ejecuta la carga de datos y A1. Faltan: "
        + ", ".join(faltantes_a4)
    )

if any(
    nombre.startswith("stock_seguridad_")
    for nombre in modelo.constraints
):
    raise ValueError(
        "A4 ya está incorporado. Para reiniciar, ejecuta A1 "
        "y después los bloques siguientes en orden."
    )

if "nivel_servicio_objetivo" not in datos.columns:
    raise ValueError(
        "Falta nivel_servicio_objetivo en inventario_inicial.csv."
    )

columnas_sigma = ["SKU"] + [f"h={t}" for t in T]

faltan_columnas = [
    columna for columna in columnas_sigma
    if columna not in tabla_c4.columns
]

if faltan_columnas:
    raise ValueError(
        "Faltan columnas en tabla_c4: "
        + ", ".join(faltan_columnas)
    )

if tabla_c4["SKU"].duplicated().any():
    raise ValueError("Hay SKU repetidos en tabla_c4.")

if set(tabla_c4["SKU"]) != set(SKUS):
    raise ValueError(
        "Los SKU de tabla_c4 no coinciden con los del modelo."
    )


# ----------------------------------------------------------------
# 3. NIVEL DE SERVICIO OBJETIVO
# ----------------------------------------------------------------

# Fuente: inventario_inicial.csv, unido a datos en A1.
# Se admite que el archivo lo exprese como 0,95 o como 95.

nivel_servicio = {}

for i in SKUS:

    valor = float(datos.loc[i, "nivel_servicio_objetivo"])

    if valor > 1:
        valor = valor / 100

    if not np.isfinite(valor) or not 0.5 <= valor < 1:
        raise ValueError(
            f"Nivel de servicio inválido para {i}: {valor}. "
            "Debe estar entre 50% y menos de 100%."
        )

    nivel_servicio[i] = valor


# ----------------------------------------------------------------
# 4. CUANTIL NORMAL CORRESPONDIENTE AL NIVEL DE SERVICIO
# ----------------------------------------------------------------

normal = NormalDist(mu=0, sigma=1)

z_servicio = {
    i: normal.inv_cdf(nivel_servicio[i])
    for i in SKUS
}

# Referencias:
# 95% -> z aproximadamente 1,645.
# 92% -> z aproximadamente 1,405.
#
# Se utilizan los niveles del archivo, no un mismo z para todos.


# ----------------------------------------------------------------
# 5. DESVIACIÓN DEL ERROR POR SKU Y HORIZONTE
# ----------------------------------------------------------------

tabla_sigma = tabla_c4.set_index("SKU")

sigma_ih = {
    (i, t): float(tabla_sigma.loc[i, f"h={t}"])
    for i in SKUS
    for t in T
}

for (i, t), valor in sigma_ih.items():

    if not np.isfinite(valor) or valor < 0:
        raise ValueError(
            f"Desviación del error inválida para {i}, "
            f"semana {t}: {valor}"
        )

# Correspondencia:
# h=1 del pronóstico -> semana 1 del plan.
# h=2 del pronóstico -> semana 2 del plan.
# ...
# h=13 del pronóstico -> semana 13 del plan.
#
# Estas desviaciones corresponden a errores de cada semana.
# No se interpretan como errores acumulados de las semanas 1 a t.
# No se suman ni se multiplican nuevamente por raíz de t.


# ----------------------------------------------------------------
# 6. CALCULAR EL STOCK DE SEGURIDAD
# ----------------------------------------------------------------

SS = {
    (i, t): z_servicio[i] * sigma_ih[i, t]
    for i in SKUS
    for t in T
}

# Unidades:
# z no tiene unidades.
# sigma está en cajas.
# Por lo tanto, SS queda expresado en cajas vendibles.
#
# No se divide SS por el rendimiento:
# s representa inventario vendible.
# La merma ya se aplica a la producción bruta en el balance de A1.
#
# No se redondea SS para los cálculos.
# Solo se redondea su presentación en las tablas.


# ----------------------------------------------------------------
# 7. INCORPORAR LAS RESTRICCIONES
# ----------------------------------------------------------------

for i in SKUS:
    for t in T:

        modelo += (
            s[i][t] >= SS[i, t],
            f"stock_seguridad_{i}_{t}"
        )

# El costo de mantener SS ya está incluido en h[i] * s[i][t],
# definido en A1. No se agrega otro costo por el mismo inventario.
#
# Tampoco se suma SS a la demanda de todas las semanas:
# es un nivel de inventario requerido, no una venta adicional.


# ----------------------------------------------------------------
# 8. TABLA RESUMEN
# ----------------------------------------------------------------

tabla_a4 = pd.DataFrame({
    "SKU": SKUS,

    "Servicio objetivo (%)": [
        100 * nivel_servicio[i]
        for i in SKUS
    ],

    "z": [
        z_servicio[i]
        for i in SKUS
    ],

    "SS mínimo (cajas)": [
        min(SS[i, t] for t in T)
        for i in SKUS
    ],

    "SS máximo (cajas)": [
        max(SS[i, t] for t in T)
        for i in SKUS
    ],

    "SS semana 13 (cajas)": [
        SS[i, 13]
        for i in SKUS
    ]
})

mostrar_tabla(
    "A4 — Stock de seguridad requerido",
    tabla_a4,
    {
        "Servicio objetivo (%)": "{:.0f}",
        "z": "{:.3f}",
        "SS mínimo (cajas)": "{:,.2f}",
        "SS máximo (cajas)": "{:,.2f}",
        "SS semana 13 (cajas)": "{:,.2f}"
    }
)


# ----------------------------------------------------------------
# 9. DETALLE DISPONIBLE PARA EL INFORME O ANEXO
# ----------------------------------------------------------------

# Se guarda sin imprimir otra tabla extensa.
# Para visualizarlo, ejecutar: display(detalle_a4)

detalle_a4 = pd.DataFrame([
    {
        "SKU": i,
        "Semana": t,
        "Servicio objetivo (%)": 100 * nivel_servicio[i],
        "z": z_servicio[i],
        "Sigma del error (cajas)": sigma_ih[i, t],
        "SS (cajas)": SS[i, t]
    }
    for i in SKUS
    for t in T
])


# ----------------------------------------------------------------
# 10. CONFIRMACIÓN
# ----------------------------------------------------------------

numero_ss = sum(
    nombre.startswith("stock_seguridad_")
    for nombre in modelo.constraints
)

assert numero_ss == len(SKUS) * len(T)

print(
    f"A4 incorporado: {numero_ss} restricciones "
    "de stock de seguridad."
)

print(
    "Política: reserva protegida al cierre semanal. "
    "Los atrasos, si existen, mantienen su penalidad."
)

print(
    "La tabla muestra requisitos de inventario, "
    "no resultados de una solución optimizada."
)

## Parte A5. Horas extra

In [ ]:
# ================================================================
# A5. HORAS EXTRA ACOTADAS Y CON COSTO
# ================================================================
#
# Variables:
# HE_linea[linea][t] = horas extra de envasado utilizadas.
# HE_mezcla[t]       = horas extra de mezcla utilizadas.
#
# Dominio:
# 0 <= HE <= máximo semanal del archivo de capacidad.
#
# Costos:
# L2:     costo_hora_extra_L2_clp.
# L1:     mismo costo que L2, como supuesto explícito.
# Mezcla: costo_hora_extra_mezcla_clp.
#
# Se amplían las restricciones de capacidad existentes en A3.
# Se conservan los consumos por producción y preparación.
#
# NO se resuelve todavía el modelo.
# ================================================================


# ----------------------------------------------------------------
# 1. VERIFICAR QUE EXISTEN LOS BLOQUES ANTERIORES
# ----------------------------------------------------------------

necesarios_a5 = [
    "modelo", "cap", "LINEAS", "T",
    "H_extra_max", "costos", "mostrar_tabla"
]

faltantes_a5 = [
    nombre for nombre in necesarios_a5
    if nombre not in globals()
]

if faltantes_a5:
    raise ValueError(
        "Primero ejecuta los bloques anteriores. Faltan: "
        + ", ".join(faltantes_a5)
    )

# Evitar agregar nuevamente variables y costos de horas extra.
if (
    "Horas extra envasado" in costos
    or "Horas extra mezcla" in costos
):
    raise ValueError(
        "A5 ya está incorporado. Para reiniciar, ejecuta A1 "
        "y luego A2–A5 en orden."
    )

# Nombres utilizados para las restricciones de A3.
nombres_capacidad_env = {
    (linea, t): f"cap_envasado_{linea}_{t}"
    for linea in LINEAS
    for t in T
}

nombres_capacidad_mezcla = {
    t: f"cap_mezcla_{t}"
    for t in T
}

restricciones_requeridas = (
    list(nombres_capacidad_env.values())
    + list(nombres_capacidad_mezcla.values())
)

faltan_restricciones = [
    nombre for nombre in restricciones_requeridas
    if nombre not in modelo.constraints
]

if faltan_restricciones:
    raise ValueError(
        "Faltan restricciones de capacidad de A3. "
        "No se puede incorporar A5 sin ellas. Faltan: "
        + ", ".join(faltan_restricciones)
    )

for nombre in restricciones_requeridas:
    if modelo.constraints[nombre].sense != pulp.LpConstraintLE:
        raise ValueError(
            f"La restricción {nombre} debe ser de tipo <=."
        )


# ----------------------------------------------------------------
# 2. LEER MÁXIMOS, TASAS Y COSTOS
# ----------------------------------------------------------------

columnas_requeridas = [
    "horas_mezcla_extra_max",
    "tasa_mezcla_L_h",
    "costo_hora_extra_L2_clp",
    "costo_hora_extra_mezcla_clp"
]

faltan_columnas = [
    columna for columna in columnas_requeridas
    if columna not in cap.columns
]

if faltan_columnas:
    raise ValueError(
        "Faltan columnas en capacidad_horizonte.csv: "
        + ", ".join(faltan_columnas)
    )

H_extra_max_mezcla = {
    t: float(cap.loc[t, "horas_mezcla_extra_max"])
    for t in T
}

tasa_mezcla = {
    t: float(cap.loc[t, "tasa_mezcla_L_h"])
    for t in T
}

# Supuesto acordado:
# el costo por hora extra de L1 se iguala al de L2.
costo_he_linea = {
    (linea, t): float(
        cap.loc[t, "costo_hora_extra_L2_clp"]
    )
    for linea in LINEAS
    for t in T
}

costo_he_mezcla = {
    t: float(
        cap.loc[t, "costo_hora_extra_mezcla_clp"]
    )
    for t in T
}

for nombre, parametro in [
    ("Máximo de horas extra de envasado", H_extra_max),
    ("Máximo de horas extra de mezcla", H_extra_max_mezcla),
    ("Costo de horas extra de envasado", costo_he_linea),
    ("Costo de horas extra de mezcla", costo_he_mezcla)
]:
    valores = np.array(
        list(parametro.values()),
        dtype=float
    )

    if not np.isfinite(valores).all() or (valores < 0).any():
        raise ValueError(
            f"{nombre}: existen valores inválidos."
        )

for t in T:
    if not np.isfinite(tasa_mezcla[t]) or tasa_mezcla[t] <= 0:
        raise ValueError(
            f"Tasa de mezcla inválida en la semana {t}."
        )


# ----------------------------------------------------------------
# 3. CREAR LAS VARIABLES DE HORAS EXTRA
# ----------------------------------------------------------------

# Son continuas: se pueden utilizar fracciones de hora.

HE_linea = pulp.LpVariable.dicts(
    "horas_extra_envasado",
    (LINEAS, T),
    lowBound=0,
    cat="Continuous"
)

HE_mezcla = pulp.LpVariable.dicts(
    "horas_extra_mezcla",
    T,
    lowBound=0,
    cat="Continuous"
)

# Cada variable recibe su máximo específico de la semana.
for linea in LINEAS:
    for t in T:
        HE_linea[linea][t].upBound = H_extra_max[linea, t]

for t in T:
    HE_mezcla[t].upBound = H_extra_max_mezcla[t]


# ----------------------------------------------------------------
# 4. REEMPLAZAR LA CAPACIDAD DE ENVASADO
# ----------------------------------------------------------------

# En A3, la restricción de envasado se expresa en HORAS:
#
# horas de producción + horas de preparación <= horas regulares
#
# Ahora:
#
# horas de producción + horas de preparación
#     <= horas regulares + horas extra
#
# Se conserva toda la expresión original y se resta HE al
# lado izquierdo. Así no se pierden los tiempos de preparación.

for linea in LINEAS:
    for t in T:

        nombre = nombres_capacidad_env[linea, t]

        restriccion_ampliada = (
            modelo.constraints[nombre].copy()
            - HE_linea[linea][t]
        )

        # Eliminar la restricción antigua antes de reemplazarla.
        del modelo.constraints[nombre]

        modelo += (
            restriccion_ampliada,
            nombre
        )


# ----------------------------------------------------------------
# 5. REEMPLAZAR LA CAPACIDAD DE MEZCLA
# ----------------------------------------------------------------

# Este bloque admite las dos representaciones de A3:
#
# A) En litros:
# litros + tasa * horas_CIP <= tasa * horas_regulares
#
# B) En horas:
# litros / tasa + horas_CIP <= horas_regulares
#
# En la versión anterior de A3, cap_mezcla estaba en LITROS.
# Por eso el valor predeterminado es "litros".
#
# Si A3 ya definió unidad_capacidad_mezcla, se respeta ese valor.

unidad_capacidad_mezcla = globals().get(
    "unidad_capacidad_mezcla",
    "litros"
)

if unidad_capacidad_mezcla not in ["litros", "horas"]:
    raise ValueError(
        "unidad_capacidad_mezcla debe ser 'litros' o 'horas'."
    )

for t in T:

    nombre = nombres_capacidad_mezcla[t]

    if unidad_capacidad_mezcla == "litros":
        capacidad_adicional = tasa_mezcla[t] * HE_mezcla[t]
    else:
        capacidad_adicional = HE_mezcla[t]

    restriccion_ampliada = (
        modelo.constraints[nombre].copy()
        - capacidad_adicional
    )

    del modelo.constraints[nombre]

    modelo += (
        restriccion_ampliada,
        nombre
    )

# A6 deberá conservar estas horas extra al introducir el
# desfase entre la preparación de jarabe y el envasado.


# ----------------------------------------------------------------
# 6. INCORPORAR LOS COSTOS
# ----------------------------------------------------------------

costos["Horas extra envasado"] = pulp.lpSum(
    costo_he_linea[linea, t] * HE_linea[linea][t]
    for linea in LINEAS
    for t in T
)

costos["Horas extra mezcla"] = pulp.lpSum(
    costo_he_mezcla[t] * HE_mezcla[t]
    for t in T
)

# Actualizar el objetivo manteniendo todos los costos anteriores.
modelo.setObjective(
    pulp.lpSum(costos.values())
)


# ----------------------------------------------------------------
# 7. COMPROBAR QUE LAS HORAS EXTRA QUEDARON INCORPORADAS
# ----------------------------------------------------------------

for linea in LINEAS:
    for t in T:

        nombre = nombres_capacidad_env[linea, t]

        coeficiente = modelo.constraints[nombre].get(
            HE_linea[linea][t],
            0
        )

        if not np.isclose(coeficiente, -1.0):
            raise ValueError(
                f"Las horas extra no quedaron correctamente "
                f"incorporadas en {nombre}."
            )

        if HE_linea[linea][t].upBound != H_extra_max[linea, t]:
            raise ValueError(
                f"Máximo incorrecto para {linea}, semana {t}."
            )

for t in T:

    nombre = nombres_capacidad_mezcla[t]

    coeficiente_esperado = (
        -tasa_mezcla[t]
        if unidad_capacidad_mezcla == "litros"
        else -1.0
    )

    coeficiente = modelo.constraints[nombre].get(
        HE_mezcla[t],
        0
    )

    if not np.isclose(coeficiente, coeficiente_esperado):
        raise ValueError(
            f"Las horas extra de mezcla no quedaron "
            f"correctamente incorporadas en la semana {t}."
        )


# ----------------------------------------------------------------
# 8. TABLA DE PARÁMETROS DE HORAS EXTRA
# ----------------------------------------------------------------

filas_a5 = []

for linea in LINEAS:

    filas_a5.append({
        "Recurso": linea,

        "Máximo semanal mínimo (h)": min(
            H_extra_max[linea, t] for t in T
        ),

        "Máximo semanal máximo (h)": max(
            H_extra_max[linea, t] for t in T
        ),

        "Costo mínimo ($/h)": min(
            costo_he_linea[linea, t] for t in T
        ),

        "Costo máximo ($/h)": max(
            costo_he_linea[linea, t] for t in T
        ),

        "Origen del costo": (
            "Supuesto: mismo costo que L2"
            if linea == "L1"
            else "Archivo de capacidad / enunciado"
        )
    })

filas_a5.append({
    "Recurso": "Mezcla",

    "Máximo semanal mínimo (h)": min(
        H_extra_max_mezcla.values()
    ),

    "Máximo semanal máximo (h)": max(
        H_extra_max_mezcla.values()
    ),

    "Costo mínimo ($/h)": min(
        costo_he_mezcla.values()
    ),

    "Costo máximo ($/h)": max(
        costo_he_mezcla.values()
    ),

    "Origen del costo": "Archivo de capacidad / enunciado"
})

tabla_a5 = pd.DataFrame(filas_a5)

mostrar_tabla(
    "A5 — Límites y costos de las horas extra",
    tabla_a5,
    {
        "Máximo semanal mínimo (h)": "{:,.1f}",
        "Máximo semanal máximo (h)": "{:,.1f}",
        "Costo mínimo ($/h)": "{:,.0f}",
        "Costo máximo ($/h)": "{:,.0f}"
    }
)


# ----------------------------------------------------------------
# 9. CONFIRMACIÓN
# ----------------------------------------------------------------

print(
    f"A5 incorporado: "
    f"{len(LINEAS) * len(T)} variables de horas extra de envasado "
    f"y {len(T)} de mezcla."
)

print(
    "Las variables tienen máximo semanal, amplían la capacidad "
    "y están incluidas en la función objetivo con su costo."
)

print(
    "Todavía no se determina cuántas horas extra se utilizarán. "
    "Ese resultado se obtiene al resolver el modelo completo."
)

## Parte A6.  Lote mínimo

In [ ]:
# ================================================================
# A6. LOTE MÍNIMO Y LEAD TIME DE JARABE DE UNA SEMANA
# ================================================================
#
# Fuentes:
# inventario_inicial.csv:
#   - lote_minimo_cajas.
#   - lead_time_jarabe_semanas.
#
# Lote mínimo:
# lote_minimo[i] * y[i][t] <= x[i][t]
#
# En conjunto con A2:
# lote_minimo[i] * y[i][t] <= x[i][t] <= M[i,t] * y[i][t]
#
# Lead time:
# q_jarabe[i][t] = litros_caja[i] * x[i][t+1]
#
# Supuestos:
# - Jarabe preparado para envasar la semana siguiente.
# - Sin almacenamiento adicional de jarabe entre semanas.
# - Disponibilidad inicial fija calculada antes de optimizar.
# - No se programa mezcla para la semana 14 en este horizonte.
#
# NO se resuelve todavía el modelo.
# ================================================================


# ----------------------------------------------------------------
# 1. VERIFICAR LOS BLOQUES ANTERIORES
# ----------------------------------------------------------------

necesarios_a6 = [
    "modelo", "datos", "SKUS", "T", "FAMILIAS",
    "skus_fam", "x", "y", "M", "SS", "s0", "b0", "d", "rend",
    "litros_caja", "tasa_mezcla", "H_reg_mezcla",
    "HE_mezcla", "w", "z", "mostrar_tabla"
]

faltantes_a6 = [
    nombre for nombre in necesarios_a6
    if nombre not in globals()
]

if faltantes_a6:
    raise ValueError(
        "Primero ejecuta A1–A5. Faltan: "
        + ", ".join(faltantes_a6)
    )

if any(
    nombre.startswith("lote_minimo_")
    for nombre in modelo.constraints
):
    raise ValueError(
        "A6 ya está incorporado. Para reiniciar, ejecuta A1 "
        "y luego los bloques siguientes en orden."
    )

for columna in [
    "lote_minimo_cajas",
    "lead_time_jarabe_semanas"
]:
    if columna not in datos.columns:
        raise ValueError(
            f"Falta {columna} en inventario_inicial.csv."
        )

# A3 debe haber vinculado las familias de mezcla con la producción.
# Esos enlaces se reemplazarán por enlaces a la semana siguiente.
enlaces_mezcla_anteriores = [
    nombre
    for nombre in modelo.constraints
    if nombre.startswith("activa_fam_lb_")
    or nombre.startswith("activa_fam_ub_")
]

if not enlaces_mezcla_anteriores:
    raise ValueError(
        "No se encontraron los enlaces de familias de mezcla de A3. "
        "Revisa ese bloque antes de ejecutar A6."
    )

for t in T:
    if f"cap_mezcla_{t}" not in modelo.constraints:
        raise ValueError(
            f"Falta la restricción cap_mezcla_{t} de A3–A5."
        )


# ----------------------------------------------------------------
# 2. LEER LOTES MÍNIMOS Y LEAD TIME
# ----------------------------------------------------------------

lote_minimo = (
    datos["lote_minimo_cajas"]
    .astype(float)
    .to_dict()
)

lead_time_jarabe = (
    datos["lead_time_jarabe_semanas"]
    .astype(float)
    .to_dict()
)

for i in SKUS:

    if (
        not np.isfinite(lote_minimo[i])
        or lote_minimo[i] <= 0
    ):
        raise ValueError(
            f"Lote mínimo inválido para {i}."
        )

    if not np.isclose(lead_time_jarabe[i], 1.0):
        raise ValueError(
            f"El lead time de {i} no es de una semana. "
            "Esta formulación está construida para lead time = 1."
        )

    if (
        not np.isfinite(litros_caja[i])
        or litros_caja[i] <= 0
    ):
        raise ValueError(
            f"Consumo de jarabe por caja inválido para {i}."
        )


# ----------------------------------------------------------------
# 3. RESTRICCIONES DE LOTE MÍNIMO
# ----------------------------------------------------------------

for i in SKUS:
    for t in T:

        modelo += (
            x[i][t] >= lote_minimo[i] * y[i][t],
            f"lote_minimo_{i}_{t}"
        )

# Si y = 0, A2 obliga a que x = 0.
# Si y = 1, A6 obliga a producir al menos el lote mínimo.
#
# Se exige un tamaño MÍNIMO de lote.
# No se exige producir múltiplos del lote mínimo.


# ----------------------------------------------------------------
# 4. FIJAR LA DISPONIBILIDAD INICIAL DE JARABE
# ----------------------------------------------------------------

# No existe un dato de jarabe inicial en los archivos utilizados.
#
# Supuesto de arranque:
# se dispone de jarabe para cubrir la necesidad neta de la semana 1,
# incluyendo la reserva de seguridad y respetando el lote mínimo.
#
# necesidad vendible =
# demanda S1 + atraso inicial + SS S1 - inventario terminado inicial.
#
# Se transforma a producción bruta dividiendo por el rendimiento.
# La disponibilidad se fija antes de resolver el modelo.

produccion_inicial_referencia = {}
J0 = {}

for i in SKUS:

    necesidad_vendible = max(
        0.0,
        d[i, 1] + b0[i] + SS[i, 1] - s0[i]
    )

    if necesidad_vendible > 0:

        produccion_bruta_requerida = (
            necesidad_vendible / rend[i]
        )

        produccion_inicial_referencia[i] = max(
            lote_minimo[i],
            produccion_bruta_requerida
        )

    else:
        produccion_inicial_referencia[i] = 0.0

    J0[i] = (
        litros_caja[i] * produccion_inicial_referencia[i]
    )

    modelo += (
        litros_caja[i] * x[i][1] <= J0[i],
        f"jarabe_inicial_{i}"
    )

# J0 está expresado en litros y no es una variable de decisión.
#
# El costo variable de la producción de la semana 1 se mantiene
# en p_var[i] * x[i][1], como en A1.
#
# No se agregan horas ni limpiezas de la semana 0, porque no
# disponemos de sus condiciones operativas.
#
# Bajo la política sin almacenamiento adicional, el jarabe inicial
# no utilizado no se transfiere automáticamente a semanas posteriores.


# ----------------------------------------------------------------
# 5. VARIABLE DE PREPARACIÓN DE JARABE
# ----------------------------------------------------------------

# q_jarabe[i][t]:
# litros preparados en la semana t y destinados al SKU i.
#
# La asignación por SKU permite contabilizar correctamente el
# consumo por formato; los CIP se agrupan por familia.

q_jarabe = pulp.LpVariable.dicts(
    "jarabe_preparado",
    (SKUS, T),
    lowBound=0,
    cat="Continuous"
)

for i in SKUS:
    for t in T:

        if t < T[-1]:

            modelo += (
                q_jarabe[i][t]
                == litros_caja[i] * x[i][t + 1],
                f"lead_time_jarabe_{i}_{t}"
            )

        else:

            # No hay demanda ni producción de semana 14
            # incorporada en este horizonte.
            modelo += (
                q_jarabe[i][t] == 0,
                f"jarabe_fin_horizonte_{i}_{t}"
            )


# ----------------------------------------------------------------
# 6. REEMPLAZAR LOS ENLACES ANTERIORES DE MEZCLA
# ----------------------------------------------------------------

# Antes, A3 vinculaba mezcla y envasado de la misma semana.
# Ahora las familias activas de mezcla dependen del envasado
# de la semana siguiente.

for nombre in enlaces_mezcla_anteriores:
    del modelo.constraints[nombre]

for familia in FAMILIAS:
    for t in T:

        if t < T[-1]:

            for i in skus_fam[familia]:

                modelo += (
                    y[i][t + 1] <= w[familia][t],
                    f"mezcla_familia_lb_{i}_{t}"
                )

            modelo += (
                w[familia][t]
                <= pulp.lpSum(
                    y[i][t + 1]
                    for i in skus_fam[familia]
                ),
                f"mezcla_familia_ub_{familia}_{t}"
            )

        else:

            modelo += (
                w[familia][t] == 0,
                f"mezcla_familia_final_{familia}_{t}"
            )

# Se conservan las restricciones de A3 que relacionan:
# - familias activas w,
# - activación del mezclador,
# - número de cambios entre familias z.
#
# Así, los CIP corresponden a las familias mezcladas esa semana,
# que abastecerán el envasado de la semana siguiente.


# ----------------------------------------------------------------
# 7. REEMPLAZAR LA CAPACIDAD DE MEZCLA
# ----------------------------------------------------------------

# Enunciado: cada cambio entre familias en mezcla consume 1,2 h.
horas_cip_mezcla = 1.2

# Se reemplaza la restricción anterior, no se mantienen ambas.
for t in T:
    del modelo.constraints[f"cap_mezcla_{t}"]

horas_uso_mezcla = {}

for t in T:

    horas_uso_mezcla[t] = (
        pulp.lpSum(
            q_jarabe[i][t] / tasa_mezcla[t]
            for i in SKUS
        )
        + horas_cip_mezcla * z[t]
    )

    modelo += (
        horas_uso_mezcla[t]
        <= H_reg_mezcla[t] + HE_mezcla[t],
        f"cap_mezcla_{t}"
    )

# La nueva restricción está expresada en horas.
unidad_capacidad_mezcla = "horas"

# Las horas extra de A5 siguen:
# - disponibles como variable de decisión;
# - acotadas por el máximo semanal;
# - incluidas en la función objetivo con su costo.
#
# El costo de CIP de A3 sigue asociado a z[t].
# No se agrega nuevamente para evitar cobrarlo dos veces.


# ----------------------------------------------------------------
# 8. TABLA DE LOTES Y CONDICIÓN INICIAL
# ----------------------------------------------------------------

tabla_a6 = pd.DataFrame({
    "SKU": SKUS,

    "Lote mínimo (cajas brutas)": [
        lote_minimo[i]
        for i in SKUS
    ],

    "Lead time (semanas)": [
        lead_time_jarabe[i]
        for i in SKUS
    ],

    "Jarabe inicial supuesto (L)": [
        J0[i]
        for i in SKUS
    ]
})

mostrar_tabla(
    "A6 — Lote mínimo y disponibilidad inicial supuesta",
    tabla_a6,
    {
        "Lote mínimo (cajas brutas)": "{:,.0f}",
        "Lead time (semanas)": "{:.0f}",
        "Jarabe inicial supuesto (L)": "{:,.2f}"
    }
)


# ----------------------------------------------------------------
# 9. COMPROBACIONES DE CONSTRUCCIÓN
# ----------------------------------------------------------------

numero_lotes = sum(
    nombre.startswith("lote_minimo_")
    for nombre in modelo.constraints
)

assert numero_lotes == len(SKUS) * len(T)

for t in T:

    coeficiente_he = modelo.constraints[
        f"cap_mezcla_{t}"
    ].get(HE_mezcla[t], 0)

    if not np.isclose(coeficiente_he, -1.0):
        raise ValueError(
            f"Las horas extra de mezcla no quedaron "
            f"incorporadas en la semana {t}."
        )

print(
    f"A6 incorporado: {numero_lotes} restricciones "
    "de lote mínimo."
)

print(
    "La mezcla de las semanas 1–12 abastece "
    "el envasado de las semanas 2–13."
)

print(
    "El envasado de la semana 1 depende del jarabe inicial "
    "supuesto que muestra la tabla."
)

print(
    "No se ha verificado la capacidad de preparación "
    "previa al inicio del horizonte."
)

print(
    "El modelo todavía no se resuelve: "
    "faltan las condiciones de A7 y A8."
)

## Parte A7.  Restricción de fin de horizonte

In [ ]:
# ================================================================
# A7. RESERVA AL FINAL DEL HORIZONTE
# ================================================================
#
# Propósito:
# evitar que el modelo reduzca artificialmente sus costos
# terminando la semana 13 sin inventario para continuar operando.
#
# Requisito del enunciado:
# incorporar una condición de fin de horizonte.
#
# Política adoptada por el equipo:
# - Todos los SKU: conservar SS de la semana 13.
# - Familias CIT y FUN: agregar una semana de cobertura,
#   estimada con el promedio del pronóstico de semanas 10–13.
#
# La cantidad de cobertura es un supuesto del modelo.
# No corresponde a una existencia ni a una exigencia numérica
# informada por la empresa.
#
# Restricción:
# s[i][13] - b[i][13] >= reserva_final[i]
#
# NO se resuelve todavía el modelo.
# ================================================================


# ----------------------------------------------------------------
# 1. VERIFICAR LOS BLOQUES ANTERIORES
# ----------------------------------------------------------------

necesarios_a7 = [
    "modelo", "SKUS", "T", "familia_de",
    "d", "SS", "s", "b", "mostrar_tabla"
]

faltantes_a7 = [
    nombre for nombre in necesarios_a7
    if nombre not in globals()
]

if faltantes_a7:
    raise ValueError(
        "Primero ejecuta los bloques anteriores. Faltan: "
        + ", ".join(faltantes_a7)
    )

if any(
    nombre.startswith("reserva_final_")
    for nombre in modelo.constraints
):
    raise ValueError(
        "A7 ya está incorporado. Para reiniciar, ejecuta A1 "
        "y luego los bloques siguientes en orden."
    )

semana_final = max(T)

if semana_final != 13:
    raise ValueError(
        "Este bloque está definido para un horizonte de 13 semanas."
    )


# ----------------------------------------------------------------
# 2. DEFINIR LA POLÍTICA DE COBERTURA FINAL
# ----------------------------------------------------------------

# Se utilizan las últimas cuatro semanas del pronóstico disponible
# para representar el nivel de demanda cercano al cierre.
#
# Promediarlas evita que una sola semana determine toda la reserva.
# Esto no sustituye un pronóstico de demanda posterior al horizonte.

semanas_referencia = [10, 11, 12, 13]

familias_con_cobertura = {"CIT", "FUN"}

# Supuesto: una semana adicional de cobertura para CIT y FUN.
semanas_cobertura_adicional = 1.0

if not set(semanas_referencia).issubset(T):
    raise ValueError(
        "Faltan semanas necesarias para calcular la cobertura final."
    )

familias_presentes = set(familia_de.values())

if not familias_con_cobertura.issubset(familias_presentes):
    raise ValueError(
        "Revisa los nombres de familia: se esperan CIT y FUN."
    )


# ----------------------------------------------------------------
# 3. CALCULAR LA DEMANDA SEMANAL DE REFERENCIA
# ----------------------------------------------------------------

demanda_referencia_final = {
    i: sum(
        d[i, t] for t in semanas_referencia
    ) / len(semanas_referencia)
    for i in SKUS
}


# ----------------------------------------------------------------
# 4. CALCULAR LA COBERTURA ADICIONAL Y LA RESERVA TOTAL
# ----------------------------------------------------------------

cobertura_final = {}
reserva_final = {}

for i in SKUS:

    if familia_de[i] in familias_con_cobertura:

        cobertura_final[i] = (
            semanas_cobertura_adicional
            * demanda_referencia_final[i]
        )

    else:
        cobertura_final[i] = 0.0

    reserva_final[i] = (
        SS[i, semana_final]
        + cobertura_final[i]
    )

    if (
        not np.isfinite(reserva_final[i])
        or reserva_final[i] < 0
    ):
        raise ValueError(
            f"Reserva final inválida para {i}."
        )

# Todas las cantidades están expresadas en cajas vendibles.
#
# No se divide la reserva por el rendimiento:
# el balance de A1 ya convierte producción bruta en vendible.
#
# No se suma esta reserva a la demanda:
# se exige como inventario remanente al cierre del horizonte.


# ----------------------------------------------------------------
# 5. INCORPORAR LAS RESTRICCIONES FINALES
# ----------------------------------------------------------------

for i in SKUS:

    modelo += (
        s[i][semana_final] - b[i][semana_final]
        >= reserva_final[i],
        f"reserva_final_{i}"
    )

# Se utiliza inventario NETO:
#
# Si hay pedidos pendientes, esas cajas se descuentan del
# inventario antes de comprobar la reserva.
#
# Así, el mismo inventario no puede contabilizarse simultáneamente
# como cobertura de atrasos y como reserva disponible futura.
#
# Esta restricción no impone por sí sola b[i][13] = 0.
# El atraso final efectivo se revisará después de resolver.


# ----------------------------------------------------------------
# 6. TABLA DE REQUISITOS FINALES
# ----------------------------------------------------------------

tabla_a7 = pd.DataFrame({
    "SKU": SKUS,

    "Familia": [
        familia_de[i]
        for i in SKUS
    ],

    "SS semana 13 (cajas)": [
        SS[i, semana_final]
        for i in SKUS
    ],

    "Cobertura adicional (cajas)": [
        cobertura_final[i]
        for i in SKUS
    ],

    "Reserva neta exigida (cajas)": [
        reserva_final[i]
        for i in SKUS
    ]
})

mostrar_tabla(
    "A7 — Reserva requerida al finalizar la semana 13",
    tabla_a7,
    {
        "SS semana 13 (cajas)": "{:,.2f}",
        "Cobertura adicional (cajas)": "{:,.2f}",
        "Reserva neta exigida (cajas)": "{:,.2f}"
    }
)


# ----------------------------------------------------------------
# 7. CONFIRMACIÓN
# ----------------------------------------------------------------

numero_reservas = sum(
    nombre.startswith("reserva_final_")
    for nombre in modelo.constraints
)

assert numero_reservas == len(SKUS)

print(
    f"A7 incorporado: {numero_reservas} restricciones "
    "de reserva final."
)

print(
    "CIT y FUN: SS final más una semana de cobertura, "
    "calculada con el promedio del pronóstico de semanas 10–13."
)

print(
    "NEC: SS final, sin cobertura adicional."
)

print(
    "La tabla muestra requisitos. El inventario y los atrasos "
    "resultantes se comprobarán al resolver después de A8."
)

## Parte A8. Pedidos en firme

In [ ]:
# ================================================================
# A8. PEDIDOS EN FIRME Y RESOLUCIÓN DEL MODELO COMPLETO
# ================================================================
#
# Fuente:
# compromisos_semana_focal.csv
#
# Requisito:
# la producción vendible de la semana 12 debe cubrir,
# como mínimo, los compromisos de cada SKU.
#
# Restricción:
# rend[i] * x[i][12] >= pedidos_firmes[i]
#
# Los compromisos NO se suman nuevamente a la demanda.
#
# Después se resuelve el modelo completo A1–A8 y se muestran
# los resultados efectivos.
# ================================================================


# ----------------------------------------------------------------
# 1. VERIFICAR LOS BLOQUES ANTERIORES
# ----------------------------------------------------------------

from time import perf_counter

necesarios_a8 = [
    "modelo", "SKUS", "LINEAS", "T",
    "compromisos", "linea_de", "rend",
    "x", "y", "s", "b", "d", "s0", "b0",
    "costos", "SS", "reserva_final", "cobertura_final",
    "HE_linea", "HE_mezcla",
    "costo_he_linea", "costo_he_mezcla",
    "lote_minimo", "M",
    "q_jarabe", "J0", "litros_caja",
    "mostrar_tabla"
]

faltantes_a8 = [
    nombre for nombre in necesarios_a8
    if nombre not in globals()
]

if faltantes_a8:
    raise ValueError(
        "Primero ejecuta A1–A7. Faltan: "
        + ", ".join(faltantes_a8)
    )

# Comprobar que están incorporados los principales bloques.
bloques_requeridos = {
    "A1": "balance_",
    "A2": "enlace_",
    "A3/A5": "cap_envasado_",
    "A4": "stock_seguridad_",
    "A6": "lote_minimo_",
    "A7": "reserva_final_"
}

for bloque, prefijo in bloques_requeridos.items():

    if not any(
        nombre.startswith(prefijo)
        for nombre in modelo.constraints
    ):
        raise ValueError(
            f"No se encontraron las restricciones de {bloque}."
        )

semana_focal = 12
semana_final = max(T)

if semana_focal not in T:
    raise ValueError(
        "La semana 12 no pertenece al horizonte del modelo."
    )


# ----------------------------------------------------------------
# 2. LEER Y VALIDAR LOS COMPROMISOS
# ----------------------------------------------------------------

columna_cantidad = "cantidad_comprometida_cajas"

columnas_requeridas = [
    "sku",
    columna_cantidad
]

faltan_columnas = [
    columna for columna in columnas_requeridas
    if columna not in compromisos.columns
]

if faltan_columnas:
    raise ValueError(
        "Faltan columnas en compromisos_semana_focal.csv: "
        + ", ".join(faltan_columnas)
    )

compromisos_a8 = compromisos.copy()

if compromisos_a8["sku"].isna().any():
    raise ValueError(
        "Hay pedidos sin SKU en el archivo de compromisos."
    )

compromisos_a8[columna_cantidad] = pd.to_numeric(
    compromisos_a8[columna_cantidad],
    errors="raise"
)

cantidades = compromisos_a8[columna_cantidad].to_numpy(
    dtype=float
)

if not np.isfinite(cantidades).all() or (cantidades < 0).any():
    raise ValueError(
        "Los compromisos contienen cantidades inválidas."
    )

sku_desconocidos = (
    set(compromisos_a8["sku"]) - set(SKUS)
)

if sku_desconocidos:
    raise ValueError(
        "Existen compromisos para SKU que no están en el modelo: "
        + ", ".join(sorted(sku_desconocidos))
    )

# Un mismo SKU puede aparecer en varias órdenes.
# Se suman sus compromisos para obtener el mínimo semanal.
pedidos_firmes = (
    compromisos_a8
    .groupby("sku")[columna_cantidad]
    .sum()
    .reindex(SKUS, fill_value=0)
    .astype(float)
    .to_dict()
)


# ----------------------------------------------------------------
# 3. RESTRICCIONES DE PRODUCCIÓN DE LA SEMANA 12
# ----------------------------------------------------------------

# Si se vuelve a ejecutar A8, se reemplazan únicamente
# sus propias restricciones de compromisos.

for nombre in list(modelo.constraints):

    if nombre.startswith("compromiso_semana12_"):
        del modelo.constraints[nombre]

for i in SKUS:

    modelo += (
        rend[i] * x[i][semana_focal]
        >= pedidos_firmes[i],
        f"compromiso_semana12_{i}"
    )

# Se exige producción VENDIBLE, no producción bruta.
#
# El inventario previo no sustituye este mínimo, porque
# el requisito se refiere expresamente a producir en semana 12.
#
# Este modelo semanal no certifica cumplimiento por día y hora.
# Eso requiere la programación detallada de la semana.


# ----------------------------------------------------------------
# 4. ACTUALIZAR LA FUNCIÓN OBJETIVO COMPLETA
# ----------------------------------------------------------------

# Incluye todos los componentes agregados en A1–A7:
# producción, inventario, atrasos, preparaciones, CIP y horas extra.

modelo.setObjective(
    pulp.lpSum(costos.values())
)

# Verificar que las horas extra tienen costo en el objetivo.
for linea in LINEAS:
    for t in T:

        coeficiente = modelo.objective.get(
            HE_linea[linea][t],
            0
        )

        if not np.isclose(
            coeficiente,
            costo_he_linea[linea, t]
        ):
            raise ValueError(
                f"Costo de horas extra incorrecto para "
                f"{linea}, semana {t}."
            )

for t in T:

    coeficiente = modelo.objective.get(
        HE_mezcla[t],
        0
    )

    if not np.isclose(
        coeficiente,
        costo_he_mezcla[t]
    ):
        raise ValueError(
            f"Costo de horas extra de mezcla incorrecto "
            f"en la semana {t}."
        )


# ----------------------------------------------------------------
# 5. RESOLVER EL MODELO COMPLETO
# ----------------------------------------------------------------

# CBC muestra el avance de la resolución.
# gapRel=0 solicita cerrar la brecha relativa;
# el solver sigue trabajando con tolerancias numéricas.

solver = pulp.PULP_CBC_CMD(
    msg=True,
    gapRel=0
)

if not solver.available():
    raise RuntimeError(
        "CBC no está disponible. Revisa la instalación "
        "de PuLP en la configuración inicial."
    )

print("Resolviendo el modelo completo A1–A8...")

inicio_resolucion = perf_counter()

modelo.solve(solver)

tiempo_resolucion = perf_counter() - inicio_resolucion

estado_modelo = pulp.LpStatus[modelo.status]

print(f"\nEstado del modelo: {estado_modelo}")
print(f"Tiempo de resolución: {tiempo_resolucion:.2f} segundos")

# No presentar resultados como óptimos si no se certificó
# una solución óptima.
if (
    modelo.status != pulp.LpStatusOptimal
    or modelo.sol_status != pulp.LpSolutionOptimal
):
    raise RuntimeError(
        "No se obtuvo una solución óptima certificada. "
        "No interpretes las tablas como resultados finales. "
        "Revisa el estado y el registro del solver."
    )


# ----------------------------------------------------------------
# 6. COMPROBAR FACTIBILIDAD NUMÉRICA
# ----------------------------------------------------------------

# Tolerancia para pequeñas diferencias de precisión del solver.
tolerancia = 0.05

restricciones_incumplidas = [
    nombre
    for nombre, restriccion in modelo.constraints.items()
    if not restriccion.valid(tolerancia)
]

variables_invalidas = [
    variable.name
    for variable in modelo.variables()
    if not variable.valid(tolerancia)
]

if restricciones_incumplidas or variables_invalidas:
    raise RuntimeError(
        "La solución presenta incumplimientos superiores "
        "a la tolerancia de comprobación.\n"
        f"Restricciones: {restricciones_incumplidas[:10]}\n"
        f"Variables: {variables_invalidas[:10]}"
    )


# ----------------------------------------------------------------
# 7. GUARDAR EL PLAN COMPLETO POR SKU Y SEMANA
# ----------------------------------------------------------------

filas_plan = []

for i in SKUS:
    for t in T:

        produccion_bruta = float(pulp.value(x[i][t]))
        produccion_vendible = rend[i] * produccion_bruta

        filas_plan.append({
            "SKU": i,
            "Línea": linea_de[i],
            "Semana": t,
            "Demanda (cajas)": d[i, t],
            "Producción bruta (cajas)": produccion_bruta,
            "Producción vendible (cajas)": produccion_vendible,
            "Merma estimada (cajas)": (
                produccion_bruta - produccion_vendible
            ),
            "Inventario final (cajas)": float(
                pulp.value(s[i][t])
            ),
            "Atraso final (cajas)": float(
                pulp.value(b[i][t])
            ),
            "SS requerido (cajas)": SS[i, t],
            "Preparación activa": int(
                round(pulp.value(y[i][t]))
            ),
            "Jarabe preparado (L)": float(
                pulp.value(q_jarabe[i][t])
            )
        })

plan_final = pd.DataFrame(filas_plan)

# plan_final conserva los resultados sin redondear.
# Las tablas siguientes solo redondean su presentación.


# ----------------------------------------------------------------
# 8. TABLA DE CUMPLIMIENTO DE LOS COMPROMISOS
# ----------------------------------------------------------------

filas_compromisos = []

for i in SKUS:

    produccion_bruta = float(
        pulp.value(x[i][semana_focal])
    )

    produccion_vendible = (
        rend[i] * produccion_bruta
    )

    filas_compromisos.append({
        "SKU": i,
        "Línea": linea_de[i],
        "Compromiso (cajas)": pedidos_firmes[i],
        "Producción bruta S12": produccion_bruta,
        "Producción vendible S12": produccion_vendible,
        "Cumple": (
            "Sí"
            if produccion_vendible + tolerancia
            >= pedidos_firmes[i]
            else "No"
        )
    })

tabla_compromisos_a8 = pd.DataFrame(filas_compromisos)

mostrar_tabla(
    "A8 — Producción y compromisos de la semana 12",
    tabla_compromisos_a8,
    {
        "Compromiso (cajas)": "{:,.0f}",
        "Producción bruta S12": "{:,.2f}",
        "Producción vendible S12": "{:,.2f}"
    }
)


# ----------------------------------------------------------------
# 9. TABLA DE RESERVA FINAL
# ----------------------------------------------------------------

filas_reserva = []

for i in SKUS:

    inventario_final = float(
        pulp.value(s[i][semana_final])
    )

    atraso_final = float(
        pulp.value(b[i][semana_final])
    )

    inventario_neto = inventario_final - atraso_final

    filas_reserva.append({
        "SKU": i,
        "Reserva exigida (cajas)": reserva_final[i],
        "Inventario final (cajas)": inventario_final,
        "Atraso final (cajas)": atraso_final,
        "Inventario neto (cajas)": inventario_neto,
        "Cumple": (
            "Sí"
            if inventario_neto + tolerancia
            >= reserva_final[i]
            else "No"
        )
    })

tabla_reserva_final = pd.DataFrame(filas_reserva)

mostrar_tabla(
    "A7 — Verificación de la reserva al finalizar la semana 13",
    tabla_reserva_final,
    {
        "Reserva exigida (cajas)": "{:,.2f}",
        "Inventario final (cajas)": "{:,.2f}",
        "Atraso final (cajas)": "{:,.2f}",
        "Inventario neto (cajas)": "{:,.2f}"
    }
)


# ----------------------------------------------------------------
# 10. HORAS EXTRA EFECTIVAMENTE UTILIZADAS
# ----------------------------------------------------------------

filas_he = []

for t in T:

    fila = {"Semana": t}

    costo_env_semana = 0.0

    for linea in LINEAS:

        horas = float(
            pulp.value(HE_linea[linea][t])
        )

        fila[f"{linea} (h)"] = horas

        costo_env_semana += (
            horas * costo_he_linea[linea, t]
        )

    horas_mezcla = float(
        pulp.value(HE_mezcla[t])
    )

    costo_mezcla_semana = (
        horas_mezcla * costo_he_mezcla[t]
    )

    fila["Mezcla (h)"] = horas_mezcla
    fila["Costo HE ($)"] = (
        costo_env_semana + costo_mezcla_semana
    )

    filas_he.append(fila)

horas_extra_semanales = pd.DataFrame(filas_he)

fila_total_he = {"Semana": "Total"}

for columna in horas_extra_semanales.columns:

    if columna != "Semana":
        fila_total_he[columna] = (
            horas_extra_semanales[columna].sum()
        )

tabla_he_final = pd.concat(
    [
        horas_extra_semanales,
        pd.DataFrame([fila_total_he])
    ],
    ignore_index=True
)

formatos_he = {
    f"{linea} (h)": "{:,.2f}"
    for linea in LINEAS
}

formatos_he.update({
    "Mezcla (h)": "{:,.2f}",
    "Costo HE ($)": "{:,.0f}"
})

mostrar_tabla(
    "A5 — Horas extra utilizadas en la solución completa",
    tabla_he_final,
    formatos_he
)


# ----------------------------------------------------------------
# 11. COSTOS DEL PLAN COMPLETO
# ----------------------------------------------------------------

costos_obtenidos = {
    nombre: float(pulp.value(expresion))
    for nombre, expresion in costos.items()
}

costo_total = float(
    pulp.value(modelo.objective)
)

if not np.isclose(
    sum(costos_obtenidos.values()),
    costo_total,
    rtol=1e-8,
    atol=1.0
):
    raise RuntimeError(
        "La suma del desglose de costos no coincide "
        "con la función objetivo."
    )

# Comprobar también el costo de HE calculado desde las horas.
costo_he_objetivo = (
    costos_obtenidos["Horas extra envasado"]
    + costos_obtenidos["Horas extra mezcla"]
)

costo_he_tabla = horas_extra_semanales["Costo HE ($)"].sum()

if not np.isclose(
    costo_he_objetivo,
    costo_he_tabla,
    rtol=1e-8,
    atol=1.0
):
    raise RuntimeError(
        "El costo de horas extra de la tabla no coincide "
        "con el incluido en la función objetivo."
    )

tabla_costos_final = pd.DataFrame([
    {
        "Concepto": nombre,
        "Costo ($)": valor
    }
    for nombre, valor in costos_obtenidos.items()
])

tabla_costos_final = pd.concat(
    [
        tabla_costos_final,
        pd.DataFrame([{
            "Concepto": "TOTAL",
            "Costo ($)": costo_total
        }])
    ],
    ignore_index=True
)

mostrar_tabla(
    "Costo del plan — Modelo completo A1–A8",
    tabla_costos_final,
    {"Costo ($)": "{:,.0f}"}
)


# ----------------------------------------------------------------
# 12. COMPROBAR EL LOTE MÍNIMO Y EL LEAD TIME
# ----------------------------------------------------------------

for i in SKUS:
    for t in T:

        produccion = float(pulp.value(x[i][t]))
        activacion = float(pulp.value(y[i][t]))

        if produccion + tolerancia < lote_minimo[i] * activacion:
            raise RuntimeError(
                f"Incumplimiento de lote mínimo: {i}, semana {t}."
            )

        if produccion > M[i, t] * activacion + tolerancia:
            raise RuntimeError(
                f"Incumplimiento del enlace de A2: {i}, semana {t}."
            )

        if t < semana_final:

            jarabe_preparado = float(
                pulp.value(q_jarabe[i][t])
            )

            jarabe_requerido = (
                litros_caja[i]
                * float(pulp.value(x[i][t + 1]))
            )

            if abs(jarabe_preparado - jarabe_requerido) > tolerancia:
                raise RuntimeError(
                    f"Incumplimiento de lead time: {i}, semana {t}."
                )

    consumo_inicial = (
        litros_caja[i] * float(pulp.value(x[i][1]))
    )

    if consumo_inicial > J0[i] + tolerancia:
        raise RuntimeError(
            f"Se supera el jarabe inicial supuesto para {i}."
        )


# ----------------------------------------------------------------
# 13. RESUMEN FINAL
# ----------------------------------------------------------------

atraso_total_final = sum(
    float(pulp.value(b[i][semana_final]))
    for i in SKUS
)

horas_extra_totales = sum(
    float(pulp.value(HE_linea[linea][t]))
    for linea in LINEAS
    for t in T
) + sum(
    float(pulp.value(HE_mezcla[t]))
    for t in T
)

print(
    "\nVerificaciones superadas: restricciones, límites de variables, "
    "lotes mínimos, lead time y consistencia de costos."
)

print(
    f"Pedidos pendientes al finalizar la semana 13: "
    f"{atraso_total_final:,.2f} cajas."
)

if horas_extra_totales < 1e-6:

    print(
        "La solución óptima no utiliza horas extra bajo los datos "
        "y supuestos incorporados. Las variables sí están disponibles "
        "y tienen costo."
    )

else:

    print(
        "La solución utiliza horas extra. "
        "Su distribución semanal y costo se muestran en la tabla A5."
    )

print(
    "El detalle por SKU y semana quedó guardado en plan_final."
)

print(
    "Los resultados están condicionados a los supuestos declarados "
    "sobre preparaciones, stock de seguridad, jarabe inicial "
    "y reserva final."
)